In [6]:
import json
from datetime import datetime
from pathlib import Path

# =======================================
# Parameter
# =======================================
for month in range(7, 11):  # Months from July to October
    input_path = Path(f"Construction_RealLife_2024_{month}.json")
    split_date = datetime(2024, month, 15, 0, 0)

    # =======================================
    # Originaldatei laden
    # =======================================

    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Positionen und Aufträge extrahieren
    alle_bestellpositionen = data.get("Bestellpositionen", [])
    alle_auftraege = data.get("Auftraege", [])

    # =======================================
    # Bestellpositionen nach Startzeit trennen
    # =======================================

    pos_before = []
    pos_after = []

    for pos in alle_bestellpositionen:
        start = datetime.fromisoformat(pos["Start"])
        if start < split_date:
            pos_before.append(pos)
        else:
            pos_after.append(pos)

    # IDs (als Strings), die in den jeweiligen Gruppen enthalten sind
    ids_before = {str(p["ID"]) for p in pos_before}
    ids_after = {str(p["ID"]) for p in pos_after}

    # =======================================
    # Aufträge filtern – nur gültige Bestellpositionen behalten
    # =======================================

    def filter_auftraege(auftraege, gültige_ids):
        neue_auftraege = []
        for auftrag in auftraege:
            bp_strings = auftrag.get("BestellpositionenStrings", [])
            # Nur gültige IDs behalten
            gefiltert = [bp for bp in bp_strings if bp in gültige_ids]
            if gefiltert:
                neuer_auftrag = auftrag.copy()
                neuer_auftrag["BestellpositionenStrings"] = gefiltert
                neue_auftraege.append(neuer_auftrag)
        return neue_auftraege

    auftraege_before = filter_auftraege(alle_auftraege, ids_before)
    auftraege_after = filter_auftraege(alle_auftraege, ids_after)

    # Auftragnummern extrahieren
    auftrag_ids_before = {a["Auftragsnummer"] for a in auftraege_before}
    auftrag_ids_after = {a["Auftragsnummer"] for a in auftraege_after}

    # =======================================
    # Hilfsfunktionen zum Filtern von Wegen
    # =======================================

    def filter_arbeitswege(arbeitswege, gültige_auftragsnummern):
        return {
            str(arbeiter_id): {
                str(auftrag_id): dist
                for auftrag_id, dist in ziele.items()
                if auftrag_id in gültige_auftragsnummern
            }
            for arbeiter_id, ziele in arbeitswege.items()
        }

    def filter_transportwege(transportwege, gültige_auftragsnummern):
        gültig = set(map(int, gültige_auftragsnummern))
        return {
            str(from_id): {
                str(to_id): dist
                for to_id, dist in to_dict.items()
                if int(to_id) in gültig
            }
            for from_id, to_dict in transportwege.items()
            if int(from_id) in gültig
        }

    # =======================================
    # Hilfsfunktion zum Neunummerieren aller IDs
    # =======================================

    def renumber_dataset(bestellpositionen, auftraege, arbeitswege, transportwege):
        # Neue IDs zuweisen
        neue_bp_id = {str(bp["ID"]): str(new_id) for new_id, bp in enumerate(bestellpositionen)}
        neue_auftrag_id = {a["Auftragsnummer"]: str(new_id) for new_id, a in enumerate(auftraege)}

        # Bestellpositionen umnummerieren (ID + Auftragsnummer aktualisieren!)
        for new_id, bp in enumerate(bestellpositionen):
            bp["ID"] = new_id
            alte_auftragsnummer = bp.get("Auftragsnummer")
            if alte_auftragsnummer in neue_auftrag_id:
                bp["Auftragsnummer"] = neue_auftrag_id[alte_auftragsnummer]

        # Aufträge umnummerieren und referenzierte Bestellpositionen aktualisieren
        for new_id, a in enumerate(auftraege):
            a["Auftragsnummer"] = str(new_id)
            a["BestellpositionenStrings"] = [neue_bp_id[bp] for bp in a["BestellpositionenStrings"]]
            a["Baustellennummer"] = new_id

        # Arbeitswege aktualisieren
        neue_arbeitswege = {
            arbeiter_id: {
                neue_auftrag_id[auftrag_id]: dist
                for auftrag_id, dist in ziele.items()
                if auftrag_id in neue_auftrag_id
            }
            for arbeiter_id, ziele in arbeitswege.items()
        }

        # Transportwege aktualisieren
        neue_transportwege = {
            neue_auftrag_id[from_id]: {
                neue_auftrag_id[to_id]: dist
                for to_id, dist in to_dict.items()
                if to_id in neue_auftrag_id
            }
            for from_id, to_dict in transportwege.items()
            if from_id in neue_auftrag_id
        }

        return bestellpositionen, auftraege, neue_arbeitswege, neue_transportwege

    # =======================================
    # Datensätze vorbereiten und neu nummerieren
    # =======================================

    arbeitswege = data.get("ArbeitswegeString", {})
    transportwege = data.get("TransportwegeString", {})

    bp_before, auf_before, aw_before, tw_before = renumber_dataset(
        pos_before, auftraege_before,
        filter_arbeitswege(arbeitswege, auftrag_ids_before),
        filter_transportwege(transportwege, auftrag_ids_before)
    )

    bp_after, auf_after, aw_after, tw_after = renumber_dataset(
        pos_after, auftraege_after,
        filter_arbeitswege(arbeitswege, auftrag_ids_after),
        filter_transportwege(transportwege, auftrag_ids_after)
    )

    # =======================================
    # Neue JSON-Dateien zusammenbauen
    # =======================================

    data_before = data.copy()
    data_before["Bestellpositionen"] = bp_before
    data_before["Auftraege"] = auf_before
    data_before["ArbeitswegeString"] = aw_before
    data_before["TransportwegeString"] = tw_before

    data_after = data.copy()
    data_after["Bestellpositionen"] = bp_after
    data_after["Auftraege"] = auf_after
    data_after["ArbeitswegeString"] = aw_after
    data_after["TransportwegeString"] = tw_after

    # =======================================
    # Speichern
    # =======================================
    # Ordner erstellen, falls er nicht existiert
    output_folder = input_path.parent / "2_piece"
    output_folder.mkdir(exist_ok=True)
    output_path_1 = output_folder /input_path.with_name(input_path.stem + "_1_2.json")
    output_path_2 = output_folder /input_path.with_name(input_path.stem + "_2_2.json")

    with open(output_path_1, "w", encoding="utf-8") as f:
        json.dump(data_before, f, indent=2, ensure_ascii=False)

    with open(output_path_2, "w", encoding="utf-8") as f:
        json.dump(data_after, f, indent=2, ensure_ascii=False)

    print(f"✅ Aufgeteilt, neu nummeriert und gespeichert als:\n- {output_path_1.name}\n- {output_path_2.name}")

✅ Aufgeteilt, neu nummeriert und gespeichert als:
- Construction_RealLife_2024_7_1_2.json
- Construction_RealLife_2024_7_2_2.json
✅ Aufgeteilt, neu nummeriert und gespeichert als:
- Construction_RealLife_2024_8_1_2.json
- Construction_RealLife_2024_8_2_2.json
✅ Aufgeteilt, neu nummeriert und gespeichert als:
- Construction_RealLife_2024_9_1_2.json
- Construction_RealLife_2024_9_2_2.json
✅ Aufgeteilt, neu nummeriert und gespeichert als:
- Construction_RealLife_2024_10_1_2.json
- Construction_RealLife_2024_10_2_2.json


In [7]:
import json
from datetime import datetime
from pathlib import Path

# =======================================
# Parameter
# =======================================
for month in range(7, 11):  # Months from July to October
    input_path = Path(f"Construction_RealLife_2024_{month}.json")

    # 3 gleich lange Zeitfenster im Monat definieren
    split_date_1 = datetime(2024, month, 11, 0, 0)  # ca. 1/3
    split_date_2 = datetime(2024, month, 21, 0, 0)  # ca. 2/3

    # =======================================
    # Originaldatei laden
    # =======================================
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    alle_bestellpositionen = data.get("Bestellpositionen", [])
    alle_auftraege = data.get("Auftraege", [])

    # =======================================
    # Bestellpositionen nach Startzeit trennen
    # =======================================
    pos_before, pos_middle, pos_after = [], [], []

    for pos in alle_bestellpositionen:
        start = datetime.fromisoformat(pos["Start"])
        if start < split_date_1:
            pos_before.append(pos)
        elif start < split_date_2:
            pos_middle.append(pos)
        else:
            pos_after.append(pos)

    ids_before = {str(p["ID"]) for p in pos_before}
    ids_middle = {str(p["ID"]) for p in pos_middle}
    ids_after = {str(p["ID"]) for p in pos_after}

    # =======================================
    # Aufträge filtern
    # =======================================
    def filter_auftraege(auftraege, gültige_ids):
        neue_auftraege = []
        for auftrag in auftraege:
            bp_strings = auftrag.get("BestellpositionenStrings", [])
            gefiltert = [bp for bp in bp_strings if bp in gültige_ids]
            if gefiltert:
                neuer_auftrag = auftrag.copy()
                neuer_auftrag["BestellpositionenStrings"] = gefiltert
                neue_auftraege.append(neuer_auftrag)
        return neue_auftraege

    auftraege_before = filter_auftraege(alle_auftraege, ids_before)
    auftraege_middle = filter_auftraege(alle_auftraege, ids_middle)
    auftraege_after = filter_auftraege(alle_auftraege, ids_after)

    auftrag_ids_before = {a["Auftragsnummer"] for a in auftraege_before}
    auftrag_ids_middle = {a["Auftragsnummer"] for a in auftraege_middle}
    auftrag_ids_after = {a["Auftragsnummer"] for a in auftraege_after}

    # =======================================
    # Hilfsfunktionen
    # =======================================
    def filter_arbeitswege(arbeitswege, gültige_auftragsnummern):
        return {
            str(arbeiter_id): {
                str(auftrag_id): dist
                for auftrag_id, dist in ziele.items()
                if auftrag_id in gültige_auftragsnummern
            }
            for arbeiter_id, ziele in arbeitswege.items()
        }

    def filter_transportwege(transportwege, gültige_auftragsnummern):
        gültig = set(map(int, gültige_auftragsnummern))
        return {
            str(from_id): {
                str(to_id): dist
                for to_id, dist in to_dict.items()
                if int(to_id) in gültig
            }
            for from_id, to_dict in transportwege.items()
            if int(from_id) in gültig
        }

    def renumber_dataset(bestellpositionen, auftraege, arbeitswege, transportwege):
        neue_bp_id = {str(bp["ID"]): str(new_id) for new_id, bp in enumerate(bestellpositionen)}
        neue_auftrag_id = {a["Auftragsnummer"]: str(new_id) for new_id, a in enumerate(auftraege)}

        for new_id, bp in enumerate(bestellpositionen):
            bp["ID"] = new_id
            alte_auftragsnummer = bp.get("Auftragsnummer")
            if alte_auftragsnummer in neue_auftrag_id:
                bp["Auftragsnummer"] = neue_auftrag_id[alte_auftragsnummer]

        for new_id, a in enumerate(auftraege):
            a["Auftragsnummer"] = str(new_id)
            a["BestellpositionenStrings"] = [neue_bp_id[bp] for bp in a["BestellpositionenStrings"]]
            a["Baustellennummer"] = new_id

        neue_arbeitswege = {
            arbeiter_id: {
                neue_auftrag_id[auftrag_id]: dist
                for auftrag_id, dist in ziele.items()
                if auftrag_id in neue_auftrag_id
            }
            for arbeiter_id, ziele in arbeitswege.items()
        }

        neue_transportwege = {
            neue_auftrag_id[from_id]: {
                neue_auftrag_id[to_id]: dist
                for to_id, dist in to_dict.items()
                if to_id in neue_auftrag_id
            }
            for from_id, to_dict in transportwege.items()
            if from_id in neue_auftrag_id
        }

        return bestellpositionen, auftraege, neue_arbeitswege, neue_transportwege

    # =======================================
    # Umnummerieren
    # =======================================
    arbeitswege = data.get("ArbeitswegeString", {})
    transportwege = data.get("TransportwegeString", {})

    bp_before, auf_before, aw_before, tw_before = renumber_dataset(
        pos_before, auftraege_before,
        filter_arbeitswege(arbeitswege, auftrag_ids_before),
        filter_transportwege(transportwege, auftrag_ids_before)
    )

    bp_middle, auf_middle, aw_middle, tw_middle = renumber_dataset(
        pos_middle, auftraege_middle,
        filter_arbeitswege(arbeitswege, auftrag_ids_middle),
        filter_transportwege(transportwege, auftrag_ids_middle)
    )

    bp_after, auf_after, aw_after, tw_after = renumber_dataset(
        pos_after, auftraege_after,
        filter_arbeitswege(arbeitswege, auftrag_ids_after),
        filter_transportwege(transportwege, auftrag_ids_after)
    )

    # =======================================
    # Neue JSON-Dateien speichern im Unterordner "3_piece"
    # =======================================
    def save_dataset(suffix, bps, aufs, aws, tws):
        output_data = data.copy()
        output_data["Bestellpositionen"] = bps
        output_data["Auftraege"] = aufs
        output_data["ArbeitswegeString"] = aws
        output_data["TransportwegeString"] = tws

        # Ordner erstellen, falls er nicht existiert
        output_folder = input_path.parent / "3_piece"
        output_folder.mkdir(exist_ok=True)

        # Speicherpfad definieren
        output_path = output_folder / f"{input_path.stem}_{suffix}_3.json"

        # Schreiben
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(output_data, f, indent=2, ensure_ascii=False)

        return output_path.name

    file1 = save_dataset("1", bp_before, auf_before, aw_before, tw_before)
    file2 = save_dataset("2", bp_middle, auf_middle, aw_middle, tw_middle)
    file3 = save_dataset("3", bp_after, auf_after, aw_after, tw_after)

    print(f"✅ Aufgeteilt, neu nummeriert und gespeichert als:\n- {file1}\n- {file2}\n- {file3}")

✅ Aufgeteilt, neu nummeriert und gespeichert als:
- Construction_RealLife_2024_7_1_3.json
- Construction_RealLife_2024_7_2_3.json
- Construction_RealLife_2024_7_3_3.json
✅ Aufgeteilt, neu nummeriert und gespeichert als:
- Construction_RealLife_2024_8_1_3.json
- Construction_RealLife_2024_8_2_3.json
- Construction_RealLife_2024_8_3_3.json
✅ Aufgeteilt, neu nummeriert und gespeichert als:
- Construction_RealLife_2024_9_1_3.json
- Construction_RealLife_2024_9_2_3.json
- Construction_RealLife_2024_9_3_3.json
✅ Aufgeteilt, neu nummeriert und gespeichert als:
- Construction_RealLife_2024_10_1_3.json
- Construction_RealLife_2024_10_2_3.json
- Construction_RealLife_2024_10_3_3.json


In [8]:
import json
from datetime import datetime
from pathlib import Path

# =======================================
# Parameter: Monate Juli bis Oktober
# =======================================
for month in range(7, 11):
    input_path = Path(f"Construction_RealLife_2024_{month}.json")

    # Vier gleichmäßige Zeitfenster definieren
    split_date_1 = datetime(2024, month, 8, 0, 0)
    split_date_2 = datetime(2024, month, 15, 0, 0)
    split_date_3 = datetime(2024, month, 22, 0, 0)

    # =======================================
    # Originaldatei laden
    # =======================================
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    alle_bestellpositionen = data.get("Bestellpositionen", [])
    alle_auftraege = data.get("Auftraege", [])

    # =======================================
    # Bestellpositionen nach Startzeit trennen (4 Teile)
    # =======================================
    pos_1, pos_2, pos_3, pos_4 = [], [], [], []

    for pos in alle_bestellpositionen:
        start = datetime.fromisoformat(pos["Start"])
        if start < split_date_1:
            pos_1.append(pos)
        elif start < split_date_2:
            pos_2.append(pos)
        elif start < split_date_3:
            pos_3.append(pos)
        else:
            pos_4.append(pos)

    ids_1 = {str(p["ID"]) for p in pos_1}
    ids_2 = {str(p["ID"]) for p in pos_2}
    ids_3 = {str(p["ID"]) for p in pos_3}
    ids_4 = {str(p["ID"]) for p in pos_4}

    # =======================================
    # Aufträge filtern
    # =======================================
    def filter_auftraege(auftraege, gültige_ids):
        neue_auftraege = []
        for auftrag in auftraege:
            bp_strings = auftrag.get("BestellpositionenStrings", [])
            gefiltert = [bp for bp in bp_strings if bp in gültige_ids]
            if gefiltert:
                neuer_auftrag = auftrag.copy()
                neuer_auftrag["BestellpositionenStrings"] = gefiltert
                neue_auftraege.append(neuer_auftrag)
        return neue_auftraege

    auftraege_1 = filter_auftraege(alle_auftraege, ids_1)
    auftraege_2 = filter_auftraege(alle_auftraege, ids_2)
    auftraege_3 = filter_auftraege(alle_auftraege, ids_3)
    auftraege_4 = filter_auftraege(alle_auftraege, ids_4)

    auftrag_ids_1 = {a["Auftragsnummer"] for a in auftraege_1}
    auftrag_ids_2 = {a["Auftragsnummer"] for a in auftraege_2}
    auftrag_ids_3 = {a["Auftragsnummer"] for a in auftraege_3}
    auftrag_ids_4 = {a["Auftragsnummer"] for a in auftraege_4}

    # =======================================
    # Hilfsfunktionen
    # =======================================
    def filter_arbeitswege(arbeitswege, gültige_auftragsnummern):
        return {
            str(arbeiter_id): {
                str(auftrag_id): dist
                for auftrag_id, dist in ziele.items()
                if auftrag_id in gültige_auftragsnummern
            }
            for arbeiter_id, ziele in arbeitswege.items()
        }

    def filter_transportwege(transportwege, gültige_auftragsnummern):
        gültig = set(map(int, gültige_auftragsnummern))
        return {
            str(from_id): {
                str(to_id): dist
                for to_id, dist in to_dict.items()
                if int(to_id) in gültig
            }
            for from_id, to_dict in transportwege.items()
            if int(from_id) in gültig
        }

    def renumber_dataset(bestellpositionen, auftraege, arbeitswege, transportwege):
        neue_bp_id = {str(bp["ID"]): str(new_id) for new_id, bp in enumerate(bestellpositionen)}
        neue_auftrag_id = {a["Auftragsnummer"]: str(new_id) for new_id, a in enumerate(auftraege)}

        for new_id, bp in enumerate(bestellpositionen):
            bp["ID"] = new_id
            alte_auftragsnummer = bp.get("Auftragsnummer")
            if alte_auftragsnummer in neue_auftrag_id:
                bp["Auftragsnummer"] = neue_auftrag_id[alte_auftragsnummer]

        for new_id, a in enumerate(auftraege):
            a["Auftragsnummer"] = str(new_id)
            a["BestellpositionenStrings"] = [neue_bp_id[bp] for bp in a["BestellpositionenStrings"]]
            a["Baustellennummer"] = new_id

        neue_arbeitswege = {
            arbeiter_id: {
                neue_auftrag_id[auftrag_id]: dist
                for auftrag_id, dist in ziele.items()
                if auftrag_id in neue_auftrag_id
            }
            for arbeiter_id, ziele in arbeitswege.items()
        }

        neue_transportwege = {
            neue_auftrag_id[from_id]: {
                neue_auftrag_id[to_id]: dist
                for to_id, dist in to_dict.items()
                if to_id in neue_auftrag_id
            }
            for from_id, to_dict in transportwege.items()
            if from_id in neue_auftrag_id
        }

        return bestellpositionen, auftraege, neue_arbeitswege, neue_transportwege

    # =======================================
    # Umnummerieren
    # =======================================
    arbeitswege = data.get("ArbeitswegeString", {})
    transportwege = data.get("TransportwegeString", {})

    bp_1, auf_1, aw_1, tw_1 = renumber_dataset(
        pos_1, auftraege_1,
        filter_arbeitswege(arbeitswege, auftrag_ids_1),
        filter_transportwege(transportwege, auftrag_ids_1)
    )

    bp_2, auf_2, aw_2, tw_2 = renumber_dataset(
        pos_2, auftraege_2,
        filter_arbeitswege(arbeitswege, auftrag_ids_2),
        filter_transportwege(transportwege, auftrag_ids_2)
    )

    bp_3, auf_3, aw_3, tw_3 = renumber_dataset(
        pos_3, auftraege_3,
        filter_arbeitswege(arbeitswege, auftrag_ids_3),
        filter_transportwege(transportwege, auftrag_ids_3)
    )

    bp_4, auf_4, aw_4, tw_4 = renumber_dataset(
        pos_4, auftraege_4,
        filter_arbeitswege(arbeitswege, auftrag_ids_4),
        filter_transportwege(transportwege, auftrag_ids_4)
    )

    # =======================================
    # Speichern in Unterordner "3_piece"
    # =======================================
    def save_dataset(suffix, bps, aufs, aws, tws):
        output_data = data.copy()
        output_data["Bestellpositionen"] = bps
        output_data["Auftraege"] = aufs
        output_data["ArbeitswegeString"] = aws
        output_data["TransportwegeString"] = tws

        output_folder = input_path.parent / "4_piece"
        output_folder.mkdir(exist_ok=True)

        output_path = output_folder / f"{input_path.stem}_{suffix}.json"
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(output_data, f, indent=2, ensure_ascii=False)

        return output_path.name

    file1 = save_dataset("1_4", bp_1, auf_1, aw_1, tw_1)
    file2 = save_dataset("2_4", bp_2, auf_2, aw_2, tw_2)
    file3 = save_dataset("3_4", bp_3, auf_3, aw_3, tw_3)
    file4 = save_dataset("4_4", bp_4, auf_4, aw_4, tw_4)

    print(f"✅ Monat {month} aufgeteilt und gespeichert:\n- {file1}\n- {file2}\n- {file3}\n- {file4}")

✅ Monat 7 aufgeteilt und gespeichert:
- Construction_RealLife_2024_7_1_4.json
- Construction_RealLife_2024_7_2_4.json
- Construction_RealLife_2024_7_3_4.json
- Construction_RealLife_2024_7_4_4.json
✅ Monat 8 aufgeteilt und gespeichert:
- Construction_RealLife_2024_8_1_4.json
- Construction_RealLife_2024_8_2_4.json
- Construction_RealLife_2024_8_3_4.json
- Construction_RealLife_2024_8_4_4.json
✅ Monat 9 aufgeteilt und gespeichert:
- Construction_RealLife_2024_9_1_4.json
- Construction_RealLife_2024_9_2_4.json
- Construction_RealLife_2024_9_3_4.json
- Construction_RealLife_2024_9_4_4.json
✅ Monat 10 aufgeteilt und gespeichert:
- Construction_RealLife_2024_10_1_4.json
- Construction_RealLife_2024_10_2_4.json
- Construction_RealLife_2024_10_3_4.json
- Construction_RealLife_2024_10_4_4.json


In [5]:
import json
from datetime import datetime
from pathlib import Path
from collections import defaultdict

# Monat auswählen
for month in range(7, 11):  # Monate von Juli bis Oktober
    input_path = Path(f"Construction_RealLife_2024_{month}.json")

    # JSON laden
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    alle_bestellpositionen = data.get("Bestellpositionen", [])
    alle_auftraege = data.get("Auftraege", [])
    arbeitswege = data.get("ArbeitswegeString", {})
    transportwege = data.get("TransportwegeString", {})

    # =======================================
    # Positionen nach Kalenderwoche gruppieren
    # =======================================
    kw_to_positions = defaultdict(list)

    for pos in alle_bestellpositionen:
        start = datetime.fromisoformat(pos["Start"])
        kw = start.isocalendar().week
        kw_to_positions[kw].append(pos)

    # =======================================
    # Hilfsfunktionen
    # =======================================
    def filter_auftraege(auftraege, gültige_ids):
        neue_auftraege = []
        for auftrag in auftraege:
            bp_strings = auftrag.get("BestellpositionenStrings", [])
            gefiltert = [bp for bp in bp_strings if bp in gültige_ids]
            if gefiltert:
                neuer_auftrag = auftrag.copy()
                neuer_auftrag["BestellpositionenStrings"] = gefiltert
                neue_auftraege.append(neuer_auftrag)
        return neue_auftraege

    def filter_arbeitswege(arbeitswege, gültige_auftragsnummern):
        return {
            str(arbeiter_id): {
                str(auftrag_id): dist
                for auftrag_id, dist in ziele.items()
                if auftrag_id in gültige_auftragsnummern
            }
            for arbeiter_id, ziele in arbeitswege.items()
        }

    def filter_transportwege(transportwege, gültige_auftragsnummern):
        gültig = set(map(int, gültige_auftragsnummern))
        return {
            str(from_id): {
                str(to_id): dist
                for to_id, dist in to_dict.items()
                if int(to_id) in gültig
            }
            for from_id, to_dict in transportwege.items()
            if int(from_id) in gültig
        }

    def renumber_dataset(bestellpositionen, auftraege, arbeitswege, transportwege):
        neue_bp_id = {str(bp["ID"]): str(new_id) for new_id, bp in enumerate(bestellpositionen)}
        neue_auftrag_id = {a["Auftragsnummer"]: str(new_id) for new_id, a in enumerate(auftraege)}

        for new_id, bp in enumerate(bestellpositionen):
            bp["ID"] = new_id
            alte_auftragsnummer = bp.get("Auftragsnummer")
            if alte_auftragsnummer in neue_auftrag_id:
                bp["Auftragsnummer"] = neue_auftrag_id[alte_auftragsnummer]

        for new_id, a in enumerate(auftraege):
            a["Auftragsnummer"] = str(new_id)
            a["BestellpositionenStrings"] = [neue_bp_id[bp] for bp in a["BestellpositionenStrings"]]
            a["Baustellennummer"] = new_id

        neue_arbeitswege = {
            arbeiter_id: {
                neue_auftrag_id[auftrag_id]: dist
                for auftrag_id, dist in ziele.items()
                if auftrag_id in neue_auftrag_id
            }
            for arbeiter_id, ziele in arbeitswege.items()
        }

        neue_transportwege = {
            neue_auftrag_id[from_id]: {
                neue_auftrag_id[to_id]: dist
                for to_id, dist in to_dict.items()
                if to_id in neue_auftrag_id
            }
            for from_id, to_dict in transportwege.items()
            if from_id in neue_auftrag_id
        }

        return bestellpositionen, auftraege, neue_arbeitswege, neue_transportwege

    # =======================================
    # Je Kalenderwoche extrahieren, umnummerieren, speichern
    # =======================================
    output_folder = input_path.parent / "weekly_split"
    output_folder.mkdir(exist_ok=True)

    for kw, pos_liste in kw_to_positions.items():
        ids_pos = {str(p["ID"]) for p in pos_liste}
        auftraege = filter_auftraege(alle_auftraege, ids_pos)
        auftrag_ids = {a["Auftragsnummer"] for a in auftraege}

        # Umnummerieren
        bps, aufs, aws, tws = renumber_dataset(
            pos_liste,
            auftraege,
            filter_arbeitswege(arbeitswege, auftrag_ids),
            filter_transportwege(transportwege, auftrag_ids)
        )

        # Speichern
        out_data = data.copy()
        out_data["Bestellpositionen"] = bps
        out_data["Auftraege"] = aufs
        out_data["ArbeitswegeString"] = aws
        out_data["TransportwegeString"] = tws

        output_path = output_folder / f"{input_path.stem}_KW{kw}.json"
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(out_data, f, indent=2, ensure_ascii=False)

        print(f"✅ Gespeichert: {output_path.name}")

✅ Gespeichert: Construction_RealLife_2024_7_KW29.json
✅ Gespeichert: Construction_RealLife_2024_7_KW27.json
✅ Gespeichert: Construction_RealLife_2024_7_KW28.json
✅ Gespeichert: Construction_RealLife_2024_7_KW31.json
✅ Gespeichert: Construction_RealLife_2024_7_KW30.json
✅ Gespeichert: Construction_RealLife_2024_8_KW32.json
✅ Gespeichert: Construction_RealLife_2024_8_KW31.json
✅ Gespeichert: Construction_RealLife_2024_8_KW34.json
✅ Gespeichert: Construction_RealLife_2024_8_KW35.json
✅ Gespeichert: Construction_RealLife_2024_8_KW33.json
✅ Gespeichert: Construction_RealLife_2024_9_KW36.json
✅ Gespeichert: Construction_RealLife_2024_9_KW37.json
✅ Gespeichert: Construction_RealLife_2024_9_KW39.json
✅ Gespeichert: Construction_RealLife_2024_9_KW35.json
✅ Gespeichert: Construction_RealLife_2024_9_KW38.json
✅ Gespeichert: Construction_RealLife_2024_9_KW40.json
✅ Gespeichert: Construction_RealLife_2024_10_KW43.json
✅ Gespeichert: Construction_RealLife_2024_10_KW42.json
✅ Gespeichert: Constructio